# Sensitivity Study: Effects of untruncation in `Bioscreen` model

This notebook gives insight to the effect of the truncation in the `Bioscreen` model that is based on Domenico solution. The truncation in the longitudinal transport term is a major difference between the `Bioscreen` and the `Anatrans` model. The truncation can cause significant deviation from the exact solution to the transport equation (implemented in the `Mibitrans` model) for small Peclet numbers (<100).

**Authors:** Jorrit Bakker, Alraune Zech

## Background

The analytical equation implemented in the `Bioscreen` model of *mibitrans* is identical to the one implemented in the excel-based tool BIOSCREEN$^{1,2}$. It is based on the *Domenico (1987)*$^3$ analytical solution for multidimensional transport of a contaminant species, but with the addition of source depletion and source superposition (to allow for more source zones). 

The equation implemented in the `Bioscreen` model reads:
$$
\begin{equation}\tag{1}
\begin{aligned}
    C(x, y, t) & = \sum_{i=1}^{n} \left\{\frac{C^*_{0,i}}{8} \exp{\left( -\gamma_s \left( t-\frac{x}{v_R}\right)\right)} \cdot \right. \\ 
    &\quad \exp \left( \frac{x\left(1-P\right)}{2\alpha_x}\right)
    \operatorname{erfc} \left(\frac{x - Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right) \cdot \\
    &\quad  \left[ \operatorname{erf} \left( \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right) - \operatorname{erf} \left( \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right) \right] \cdot \\
    & \quad \left.\left[ \operatorname{erf} \left(\frac{Z}{2\sqrt{\alpha_z x)}} \right) - \operatorname{erf} \left( \frac{-Z}{2\sqrt{\alpha_z x}} \right) \right] \right\} 
\end{aligned}
\end{equation}
$$
with $P = \sqrt{1+4 \mu \alpha_x/v_R}$.

This equation contains a truncation in the longitudinal transport term. The untruncated form (implemented in the `Anatrans` model) reads:

$$
\begin{equation}\tag{2}
\begin{aligned}
    C(x,y,t) &= \sum_{i=1}^{n}\left\{ \frac{C^*_{0,i}}{8} \exp \left(-\gamma_s t\right)\cdot \right.  \\ 
    &\quad \left[\exp \left( \frac{x\left(1-\tilde P\right)}{2\alpha_x}\right) \right. \cdot \operatorname{erfc} \left( \frac{x - v_Rt\tilde P}{2\sqrt{\alpha_x v_Rt }} \right)\\
    &\quad \ \, +  \left.\exp \left( \frac{x\left(1+\tilde P\right)}{2\alpha_x}\right) \cdot \operatorname{erfc} \left( \frac{x + v_Rt\tilde P}{2\sqrt{\alpha_x v_Rt }} \right) \right]\cdot \\
    &\quad \left[ \operatorname{erf} \left( \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right) - \operatorname{erf} \left( \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right) \right]\cdot \\
    &\quad  \left. \left[ \operatorname{erf} \left( \frac{Z}{2\sqrt{\alpha_z x)}} \right) - \operatorname{erf} \left( \frac{-Z}{2\sqrt{\alpha_z x}} \right) \right] \right\} 
\end{aligned}
\end{equation}
$$
with $\tilde P = \sqrt{1+4 (\mu-\gamma_s) \alpha_x/v_R}$.

In case of no source depletion $\gamma_s = 0$, the truncation (i.e. cut off of term in $1+\tilde P$), is the only difference between Equations (1) and (2). Note that exact model (implemented in the `Mibitrans` model class, based on the Wexler (1992)$^4,5$-solution) still give more accurate results than the `Anatrans` or `Bioscreen` models, regardless of truncation (*West et al., 2007*$^6$). 

In the following, we will quantify the differences between the truncated and untruncated equations, and briefly discusses the implementation in the *mibitrans* package. For ease of explanation, consider a situation with no decay or source depletion, with $\mu = 0$ and $\gamma_s = 0$, thus $P =\tilde P = 1$. Differences between the two models as result of source depletion handling are explained and investigated in the notebook `sensitivity_source_depletion.ipynb`.

We simplify settings to focus solely on the effect of term truncation. We disable transverse dispersion, source depletion, and decay (i.e. parameters are chosen so that these processes do not contribute) and have only a single source zone. In this case, Equation 1 reduces to:

\begin{equation}\tag{3}
    C(x, t) = \frac{C_{0}}{2} \operatorname{erfc} \left( \frac{x - vt}{2\sqrt{\alpha_x vt }}\right)
\end{equation}

And equation 2 reduces to:

\begin{equation}\tag{4}
    C(x, t) = \frac{C_{0}}{2} \left[ \operatorname{erfc} \left( \frac{x - vt}{2\sqrt{\alpha_x vt }}\right) + \exp \left(\frac{x}{2\alpha_x}\right) \cdot \operatorname{erfc} \left( \frac{x + vt}{2\sqrt{\alpha_x vt }} \right) \right]
\end{equation}

The boundary value for continuous input is $C(x=0, t) = C_0$. However, when evaluated at $x=0$ and $t=0$, the erfc term in equation 3 resolves to $1$ and thus $C(x=0, t=0) = \frac{C_0}{2}$. Therefore, equation 3 and by extension equation 1 do not abide by the boundary condition. This is visualized below.



### Compuational challanges

In general, implementations of the error-function and the complementary error function are broadly available in python. Also computation time is no issue nowadays when it comes to the term truncation. 

However, the truncated terms shows some special behavior. Looking at the arguments $a_1 =x/\alpha_x$ ($\tilde P = 1$) in the exponential term and $a_2 = \frac{x + v_Rt}{2\sqrt{\alpha_x v_R t}}$ in the complementary error-function term, we find that that $\exp(a_1) \to \infty$ and $\text{erfc}(a_2) \to 0$ for $x \to \infty$. However, this does not necessarily mean that the additional term is negligibly small but the produce of both terms is indeterminate. When using regular implementations of `exp` and `erfc`, the calculation may cause an underflow (value < $10^{-300}$ and therefore set to 0), or overflow (value > $10^{+300}$ and therefore set to infinity) to occur. Note that 64-bit float numpy arrays are used in *mibitrans* for the calculations, consequently, it only can hold values between approximately $10^{-300}$ and $10^{+300}$.

To combat this, the `erfcx` function is used, where $\text{erfcx}(a) = \exp(a^2) \cdot \operatorname{erfc}(a)$. This transforms the additional term as: $\exp(a_1) \cdot \text{erfc}(a_2) = \frac{\exp(a_1)}{\exp(a_2^2)} \cdot \exp(a_2^2) \cdot \operatorname{erfc}(a_2) = \exp(a_1 - a_2^2) \cdot \text{erfcx}(a_2)$. Under any reasonable field conditions, this expression of the additional term will not cause over- or underflow. And is therefore implemented as such in the `Anatrans` model.


### References

$^1$ Newell, C. J., R. K. McLeod, and J. R. Gonzales (1996), BIOSCREEN Natural Attenuation Decision Support System User’s Manual Version 1.3, Tech. Rep. EPA/600/R-96/087, US EPA

$^2$ Newell, C. J., McLeod, R. K., & Gonzales, J. R. (1997), BIOSCREEN natural attenuation decision support system version 1.4 revisions, Tech. rep., U.S. EPA.

$^3$ [Domenico, P. A., 1987, An analytical model for multidimensional transport of a decaying contaminant
species, Journal of Hydrology, 91 (1), 49–58.](https://doi.org/10.1016/0022-1694(87)90127-2)

$^4$ Wexler, E. J., (1992),  Analytical solutions for one-, two-, and three-dimensional solute transport in ground-water systems with uniform flow, Tech. Rep. 03-B7, U.S. G.P.O.

$^5$[Karanovic, M., C. J. Neville, and C. B. Andrews,(2007), BIOSCREEN-AT: BIOSCREEN with an exact analytical solution, Groundwater, 45 (2), 242–245.](https://doi.org/doi:10.1111/j.1745-6584.2006.00296.x)


$^6$ [West, M. R., B. H. Kueper, and M. J. Ungs, (2007), On the use and error of approximation in the Domenico (1987) solution, Groundwater, 45 (2), 126–135.](https://doi.org/10.1111/j.1745-6584.2006.00280.x)

## Simulation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erfc
from scipy.special import erfcx
import mibitrans as mbt

### Parameter Definition and Model Setup

In [ ]:
hydro = mbt.HydrologicalParameters(
    velocity=0.1,
    porosity=0.3,
    alpha_x=1,
    alpha_y=0,
    alpha_z=0,
)

att = mbt.AttenuationParameters(
    retardation=1,
)

source = mbt.SourceParameters(
    source_zone_boundary=np.array([5,10,15]),
    source_zone_concentration=np.array([15,10,5]),
    total_mass="inf",
    depth=10,
)

model = mbt.ModelParameters(
    model_length=175,
    model_width=50,
    model_time=5*365,
    dx=1,
    dy=5,
    dt=1,
)

In [ ]:
ana_obj = mbt.Anatrans(hydro,att, source, model)
ana_ndeg = ana_obj.run()

In [ ]:
ana_ndeg.centerline(time=73, color="green", label="t = 73 days")
ana_ndeg.centerline(time=365, color="blue", label="t = 365 days")
ana_ndeg.centerline(time=3*365, color="red", label="t = 3*365 days")
plt.title("Anatrans concentration distribution at various times")
plt.legend()
plt.show()

### Visualization of the additional term

Consider the following parameters as example field conditions

In [ ]:
xxx = ana_obj.xxx
ttt = ana_obj.ttt
rv = ana_obj.rv
alpha_x = hydro.alpha_x
velocity = hydro.velocity

# Calculate additional term under no decay conditions
erfc_inner = (xxx + rv * ttt) / (2 * np.sqrt(alpha_x * rv * ttt))
add_term = np.exp(xxx / alpha_x - erfc_inner**2) * erfcx(erfc_inner)
# Calculate the advection term
advec_term = erfc((xxx - rv * ttt) / (2 * np.sqrt(alpha_x * rv * ttt)))

# Time does not evaluate at t=0, so add manually for visualization
t = np.zeros(len(ttt[:,0,0])+1)
t[1:] = ttt[:,0,0]
adv_x0 = np.zeros(len(ttt[:,0,0])+1)
add_x0 = np.zeros(len(ttt[:,0,0])+1)
# For x=0 and t=0, terms are equal to 1
adv_x0[0] = 1
add_x0[0] = 1
adv_x0[1:] = advec_term[:,0,0]
add_x0[1:] = add_term[:,0,0]

plt.plot(t, adv_x0, color="blue", label="advective term")
plt.plot(t, add_x0, color="green", label="additional term")
plt.plot(t, adv_x0 + add_x0, linestyle=":", color="black", label="sum")
plt.xlabel("time [days]")
plt.ylabel("value at x=0 [-]")
plt.legend()
plt.show()


Thus, only at around t = 150 days is the advective term is equal to $2$, and the boundary condition at $x=0$ satisfied. When including the additional term, the boundary condition is satisfied at all times. However, the error introduced by the truncation propagates, as shown in the figures below.

In [ ]:
plt.pcolormesh(xxx[None, None, :], ttt[:, None, None], add_term[:, 0, :])
plt.xlabel("x-position [m]")
plt.ylabel("time [days]")
plt.colorbar(label = "value additional term [-]")
plt.show()

x = xxx[0, 0, :]
max_additional_x_term = np.max(add_term[:,0,:], axis = 0)
plt.plot(x, max_additional_x_term, color="black")
plt.xlabel("time [days]")
plt.ylabel("Maximum value additional term [-]")
plt.show()

The value of the additional term is highest at $x=0$ and $t=0$. While the additional term decreases rapidly for higher values of $x$ and $t$, it remains relevant along the advective front. The size of the error that the truncation introduces is dependent on the Peclet number; $Pe = \frac{vx}{D_x}$. And thus for input parameters, dependent on the flow velocity and longitudinal dispersivity.